# OSL Baseline (minimal) — Gradient-only Chemotaxis (no neural network)

The **simplest possible** algorithmic controller for the 2D odor-source-localization task — **no neural network, no training**. It reads only the two sensor readings (`c_left`, `c_right`) and the temporal log-gradient (`dlog`) from the same observation the connectome / GRU policies see, and applies a small set of hand rules:

1. **Bilateral gradient steering** — turn the body toward whichever antenna reads higher concentration: `body_omega = tanh(STEER_GAIN · asym)`, where `asym = (c_L − c_R)/(c_L + c_R) ∈ [−1, 1]` is the normalized left/right asymmetry. The `tanh` keeps the command in `[−1, 1]` and, at large gain, acts as a soft sign().
2. **Stop / Go** — drive forward at full speed when the average concentration is *rising* (`dlog ≥ 0`); otherwise **stop** (no forward motion).
3. **Active sensing (optional, toggleable)** — when stopped because the signal is falling, optionally **sweep the head left/right** (`head_omega`, alternating) to actively re-acquire the gradient. Toggle with `ACTIVE_SENSING`. With `ACTIVE_SENSING=False` the head is never moved (`head_omega = 0`) — the pure gradient+stop/go lower bound.

It is given **no source location, distance, or bearing** — only the sensor readings — so it is a fair sensor-only comparison group for the learned policies.

> **반드시 성공 (clean field)**: in the noise-free (stage 0) Gaussian field the bilateral gradient is exact, so even the stripped-down rule solves the task on **every** episode. Under turbulent fields (stage 1/2) it degrades gracefully; the noise sweep below is the sensor-only baseline the learned policies are measured against.

In [1]:
# Setup — works locally and on Colab. Re-running is safe.
import os, sys, subprocess

if os.path.isdir('/content'):
    # --- Colab: clone the repo and cd in ---
    REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
    REPO_DIR = '/content/2d-osl'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    # --- Local: find the repo root (the dir containing `src/`) by walking up ---
    REPO_DIR = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_DIR, 'src')) and REPO_DIR != os.path.dirname(REPO_DIR):
        REPO_DIR = os.path.dirname(REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for p in ('assets/connectome/weights.csv', 'assets/connectome/metadata.csv'):
    assert os.path.exists(p), f'Missing {p} — wrong cwd or clone failed?'
print('repo:', REPO_DIR, '\ncwd :', os.getcwd())

repo: /home/hyunseo/Personal_Research/OSL 
cwd : /home/hyunseo/Personal_Research/OSL


In [2]:
# ===== Local setup (skip the Colab cell above; run this instead) =====
import os, sys

REPO_DIR = os.path.abspath('..')  # notebook lives at <repo>/ipynb/, so .. is the repo root
if not os.path.isdir(os.path.join(REPO_DIR, 'src', 'baselines')):
    REPO_DIR = os.path.abspath('.')
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('repo:', REPO_DIR, '\ncwd :', os.getcwd())

repo: /home/hyunseo/Personal_Research/OSL 
cwd : /home/hyunseo/Personal_Research/OSL


## Configuration
Same env knobs as the PPO / SAC notebooks so results are directly comparable. The controller has exactly **one** knob — `STEER_GAIN` — and a stateless `minimal_act(obs)` function (defined here, used by the smoke check and every eval below).

In [3]:
# ===== HYPERPARAMETERS — edit freely =====
# Env: the SAME OslEnv as the PPO/GRU and SAC notebooks. The observation
# ([c_left, c_right, dlog, prev_v, prev_body_omega, prev_head_omega]), the
# action space ([v, body_omega, head_omega] in [-1,1]), and all movement limits
# (v_max, body/head omega max, stop_threshold) come from EnvConfig defaults —
# identical for every agent — so this baseline is a fair sensor-only comparison.
# Active sensing here is the SAME event the PPO/GRU notebooks count: a head
# sweep while the body is stopped (env flag event_is_high_cast_like).
ENV_KW = dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)

# --- Steering: the main success driver ---
# `asym` is already normalized to [-1, 1], so tanh(STEER_GAIN * asym) squashes
# the steering command back into [-1, 1] (no clip). STEER_GAIN sets how fast it
# saturates; the bilateral difference is tiny (spacing 0.15 mm << sigma 30 mm),
# so a large gain is needed. At 80 tanh is effectively a soft sign() and the
# clean field is solved every episode.
STEER_GAIN = 80.0

# --- Active sensing ---
# When the body has stopped (signal falling) the head sweeps left/right —
# this is the active-sensing event the PPO/GRU notebooks count
# (env flag `event_is_high_cast_like`). It is always on for this baseline.
ACTIVE_SENSING = True
AS_HEAD_OMEGA = 1.0   # head-sweep amplitude (action[2]) during active sensing
AS_HALF_PERIOD = 6    # steps before flipping sweep direction

# Evaluation
EVAL_SEED_BASE = 20_000
EVAL_EPISODES = 100        # episodes per (stage, strength) condition
EVAL_NOISE_STAGE = 2       # for the elite-seed GIF search
EVAL_NOISE_STRENGTH = 1.0
# ==========================================

import numpy as np
from src.envs.osl_env import EnvConfig, OslEnv


class MinimalController:
    """Stateless gradient steering + stop/go, with optional active sensing
    (head sweep) when stopped. The only internal state is the head-sweep
    phase/direction; set active_sensing=False for the pure gradient lower bound.
    """
    def __init__(self, active_sensing=ACTIVE_SENSING, steer_gain=STEER_GAIN,
                 as_head_omega=AS_HEAD_OMEGA, as_half_period=AS_HALF_PERIOD):
        self.active_sensing = active_sensing
        self.steer_gain = steer_gain
        self.as_head_omega = as_head_omega
        self.as_half_period = as_half_period
        self.reset()

    def reset(self):
        self._sweep_dir = 1.0
        self._sweep_phase = 0

    def act(self, obs):
        c_left, c_right, dlog = float(obs[0]), float(obs[1]), float(obs[2])
        denom = c_left + c_right + 1e-9
        asym = (c_left - c_right) / denom
        # Rule 1 — steer toward the stronger antenna (soft sign at large gain).
        body_omega = float(np.tanh(self.steer_gain * asym))
        rising = dlog >= 0.0
        if rising:
            # Rule 2 — go: full speed, no head sweep, reset the sweep phase.
            self._sweep_phase = 0
            v_action = 1.0                       # v=1 -> [0,1]->[-1,1] => +1
            head_omega = 0.0
            return np.asarray([v_action, body_omega, head_omega], dtype=np.float32)
        # Falling -> stop. Rule 3 — optionally sweep the head (active sensing).
        v_action = -1.0                          # v=0 -> stop
        head_omega = 0.0
        if self.active_sensing:
            self._sweep_phase += 1
            if self._sweep_phase >= self.as_half_period:
                self._sweep_phase = 0
                self._sweep_dir *= -1.0
            head_omega = float(np.clip(self._sweep_dir * self.as_head_omega, -1.0, 1.0))
        return np.asarray([v_action, body_omega, head_omega], dtype=np.float32)


def make_env(stage, strength, seed):
    cfg = {**ENV_KW, 'noise_stage': int(stage), 'noise_strength': float(strength), 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg))


def run_episode(env, controller, seed, collect_traj=False, render_fn=None):
    """Roll out one episode. Returns {seed, return, success, steps, as_steps[, frames]}.
    `casts` = number of steps the env labeled as a head-cast (event_is_high_cast_like).
    """
    obs, _ = env.reset(seed=seed)
    controller.reset()
    ret, success, as_steps = 0.0, False, 0
    traj_x, traj_y, as_x, as_y, frames = [], [], [], [], []
    for t in range(env.max_steps):
        action = controller.act(obs)
        if collect_traj:
            traj_x.append(env.x_mm); traj_y.append(env.y_mm)
        obs, r, term, trunc, info = env.step(action)
        ret += float(r)
        if info.get("event_is_high_cast_like"):
            as_steps += 1
            if collect_traj:
                as_x.append(env.x_mm); as_y.append(env.y_mm)
        if collect_traj and render_fn is not None:
            frames.append(render_fn(env, traj_x, traj_y, as_x, as_y, t,
                                    title=f"minimal seed={seed} step={t} AS={as_steps}"))
        if term or trunc:
            success = bool(info.get("success", False))
            break
    return {"seed": seed, "return": ret, "success": success, "steps": t + 1,
            "as_steps": as_steps, "frames": frames if collect_traj else None}


# Default controller used by the smoke check / single-condition evals below.
controller = MinimalController(active_sensing=ACTIVE_SENSING)
print(f'minimal controller ready: STEER_GAIN={STEER_GAIN}  ACTIVE_SENSING={ACTIVE_SENSING}')

minimal controller ready: STEER_GAIN=80.0  ACTIVE_SENSING=True


## Smoke check

In [6]:
env = OslEnv()
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'action_space', env.action_space.shape)

a = controller.act(obs)
print('action', a, '(continuous [v, body_omega, head_omega] in [-1, 1])')
r = run_episode(env, controller, seed=0)
print('one clean episode →', {k: r[k] for k in ('return', 'success', 'steps', 'as_steps')})

obs (6,) action_space (3,)
action [ 1.        -0.1491714  0.       ] (continuous [v, body_omega, head_omega] in [-1, 1])
one clean episode → {'return': 21.227848215568244, 'success': True, 'steps': 509, 'as_steps': 0}


In [15]:
# ===== Shared plot style — matches presentation_assets/ =====
# Keep every figure in this notebook visually consistent with the slide assets:
# matplotlib defaults (DejaVu Sans), simple short titles, clean axes,
# saturated presentation colors, and point+line plots. Run this once before
# any plotting cell below.
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.titleweight': 'normal',
    'axes.labelsize': 12,
    'axes.grid': False,
    'axes.linewidth': 1.1,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 3.0,
    'lines.markersize': 9,
    'legend.frameon': False,
    'figure.dpi': 110,
    'savefig.dpi': 150,
})

PRESENTATION_BLUE = '#1f77b4'
PRESENTATION_RED = '#d62728'
ACCENT = PRESENTATION_BLUE
AS_COLORS = {'active sensing ON': PRESENTATION_RED,
             'active sensing OFF': PRESENTATION_BLUE}
POINT_LINE = dict(marker='o', linewidth=3.0, markersize=9)
print('plot style set to match presentation_assets/')


plot style set to match presentation_assets/


## Evaluation — success rate, steps-to-source
Runs `EVAL_EPISODES` episodes at a chosen `(stage, strength)` and reports success rate + step statistics. The clean field (stage 0) should be **100%**.

In [ ]:
import numpy as np
from src.utils.baseline_plots import seed_subaggregates

# Per-condition episode budget, split across SEEDS so each seed group is a
# sub-aggregate (→ grey dots) while the pooled mean is the colored point.
# Bumped to 200 for stable success-rate statistics across every figure.
EVAL_EPISODES = 200
EVAL_SEEDS = [0, 1, 2, 3]        # seed groups for the grey per-seed dots


def evaluate(stage, strength, ctrl=None, n_episodes=EVAL_EPISODES,
             seeds=tuple(EVAL_SEEDS), seed_base=EVAL_SEED_BASE):
    """Run `n_episodes` split evenly across `seeds`. Returns pooled scalars plus
    per-seed sub-aggregates (success rate, mean steps-on-success) for grey dots.
    """
    ctrl = ctrl if ctrl is not None else controller
    per_seed = int(np.ceil(n_episodes / len(seeds)))
    succ, steps, rets, as_list, ep_seed = [], [], [], [], []
    for gi, sd in enumerate(seeds):
        for k in range(per_seed):
            seed = seed_base + gi * 100_000 + k
            env = make_env(stage, strength, seed)
            r = run_episode(env, ctrl, seed=seed)
            succ.append(int(r['success'])); steps.append(r['steps'])
            rets.append(r['return']); as_list.append(r['as_steps']); ep_seed.append(sd)
    succ = np.asarray(succ); steps = np.asarray(steps)
    as_arr = np.asarray(as_list); ep_seed = np.asarray(ep_seed)
    succ_steps = steps[succ == 1]
    # Per-seed sub-aggregates (the grey-dot distribution shown on every plot).
    seed_success = seed_subaggregates(succ, ep_seed, reducer=np.mean)
    seed_steps = []
    for sd in seeds:
        m = (ep_seed == sd) & (succ == 1)
        seed_steps.append(float(steps[m].mean()) if m.any() else np.nan)
    seed_steps = np.asarray(seed_steps, dtype=float)
    return {
        'stage': stage, 'strength': strength, 'n': len(succ),
        'success_rate': float(succ.mean()),
        'mean_steps_all': float(steps.mean()),
        'mean_steps_success': float(succ_steps.mean()) if len(succ_steps) else float('nan'),
        'mean_return': float(np.mean(rets)),
        'active_sensing_fraction': float((as_arr / np.maximum(steps, 1)).mean()),
        'seed_success_rate': seed_success,          # 1 value/seed → grey dots
        'seed_mean_steps_success': seed_steps,      # 1 value/seed → grey dots
    }

clean = evaluate(0, 0.0)
print(f"CLEAN field (stage 0): success={clean['success_rate']:.0%}  "
      f"mean_steps(success)={clean['mean_steps_success']:.0f}  "
      f"mean_return={clean['mean_return']:.2f}  (n={clean['n']})")
assert clean['success_rate'] == 1.0, 'Clean field should be solved on every episode!'
print('✅ minimal baseline solves the clean field on every episode (반드시 성공).')

## Elite-seed GIF
Renders one trajectory (same look as the PPO/SAC notebooks). With the same seed the trajectory + plume are fully reproducible.

In [13]:
import os
from IPython.display import Image as DisplayImage, display
from src.utils.plotter import render_rollout_frame, save_gif

# Default to a mild-noise condition where the baseline reliably succeeds.
GIF_STAGE = 2
GIF_STRENGTH = 1.0
GIF_SEED = None            # None = auto-pick the first successful episode; or set an int

chosen = GIF_SEED
if chosen is None:
    chosen = EVAL_SEED_BASE
    for i in range(200):
        s = EVAL_SEED_BASE + i
        env = make_env(GIF_STAGE, GIF_STRENGTH, s)
        rr = run_episode(env, controller, seed=s)
        if rr['success']:
            chosen = s; break
print('rendering seed', chosen)

env = make_env(GIF_STAGE, GIF_STRENGTH, chosen)
result = run_episode(env, controller, seed=chosen, collect_traj=True, render_fn=render_rollout_frame)
print(f"seed={result['seed']} return={result['return']:.2f} "
      f"success={result['success']} steps={result['steps']} AS_steps={result['as_steps']}")

os.makedirs('runs/baseline_minimal/plots', exist_ok=True)
gif_path = f'runs/baseline_minimal/plots/agent_seed{chosen}.gif'
save_gif(result['frames'], gif_path, fps=15)
display(DisplayImage(data=open(gif_path, 'rb').read(), format='gif'))

KeyboardInterrupt: 

## Noise sweep — robustness curve
Success rate across the full curriculum (`stage` × `strength`). The clean field is 100%; turbulence degrades it. Compare directly against the full chemotaxis baseline — the gap is what cast recovery (and learning) buys.

In [ ]:
import json
import matplotlib.pyplot as plt
from src.utils.baseline_plots import apply_style, point_line_with_seeds, ACCENT

apply_style()

SWEEP = [
    (0, 0.0), (1, 0.3), (1, 0.6), (1, 1.0),
    (2, 0.3), (2, 0.6), (2, 1.0),
]
SWEEP_EPISODES = EVAL_EPISODES   # 200 per condition

rows = [evaluate(stage, strength, n_episodes=SWEEP_EPISODES) for stage, strength in SWEEP]

print(f"{'stage':>5} {'α':>4} {'success':>8} {'steps(succ)':>12} {'return':>8}")
for r in rows:
    print(f"{r['stage']:>5} {r['strength']:>4.1f} {r['success_rate']:>7.0%} "
          f"{r['mean_steps_success']:>12.0f} {r['mean_return']:>8.2f}")

labels = [f"s{r['stage']}·α{r['strength']}" for r in rows]
x = np.arange(len(labels))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# scalar-per-condition → mean point+line (colored) + per-seed grey dots
point_line_with_seeds(ax[0], x, [r['seed_success_rate'] for r in rows], color=ACCENT)
ax[0].set_ylim(-0.02, 1.04); ax[0].set_ylabel('success rate'); ax[0].set_title('Success ratio')
point_line_with_seeds(ax[1], x, [r['seed_mean_steps_success'] for r in rows], color=ACCENT)
ax[1].set_ylabel('steps to source'); ax[1].set_title('Steps to source')
for a_ in ax:
    a_.set_xticks(x); a_.set_xticklabels(labels, rotation=30)
fig.tight_layout()
os.makedirs('runs/baseline_minimal', exist_ok=True)
fig.savefig('runs/baseline_minimal/noise_sweep.png', dpi=150)
with open('runs/baseline_minimal/noise_sweep.json', 'w') as f:
    json.dump([{k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()} for r in rows],
              f, indent=2)
plt.show()
print('\n[saved] runs/baseline_minimal/noise_sweep.{png,json}')

## Average steps-to-goal on successful episodes

For each noise condition, the **average number of steps to reach the source** — computed **only over episodes that succeeded** (failed episodes run out the clock and would bias the mean).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from src.utils.baseline_plots import apply_style, point_line_with_seeds, ACCENT

apply_style()

STEPS_CONDITIONS = [
    (0, 0.0), (1, 0.3), (1, 0.6), (1, 1.0),
    (2, 0.3), (2, 0.6), (2, 1.0),
]
STEPS_EPISODES = EVAL_EPISODES   # 200 per condition

# Reuse `evaluate` so the steps-to-goal figure shares the SAME per-seed
# sub-aggregates (grey dots) and pooled mean (colored point) as every other
# scalar-per-condition figure — one consistent format.
steps_rows = [evaluate(stage, strength, n_episodes=STEPS_EPISODES)
              for stage, strength in STEPS_CONDITIONS]

print(f"Avg steps-to-goal over SUCCESSFUL episodes ({STEPS_EPISODES} eps/condition)\n")
print(f"{'stage':>5} {'α':>4} {'success':>8} {'avg steps(succ)':>16}")
for r in steps_rows:
    print(f"{r['stage']:>5} {r['strength']:>4.1f} {r['success_rate']:>7.0%} "
          f"{r['mean_steps_success']:>16.1f}")

labels = [f"s{r['stage']}·α{r['strength']}" for r in steps_rows]
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 4.5))
point_line_with_seeds(ax, x, [r['seed_mean_steps_success'] for r in steps_rows], color=ACCENT)
ax.set_ylabel('steps to source'); ax.set_title('Steps to source')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30)
fig.tight_layout()
os.makedirs('runs/baseline_minimal', exist_ok=True)
fig.savefig('runs/baseline_minimal/steps_to_goal.png', dpi=150)
with open('runs/baseline_minimal/steps_to_goal.json', 'w') as f:
    json.dump([{k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()}
               for r in steps_rows], f, indent=2)
plt.show()
print('\n[saved] runs/baseline_minimal/steps_to_goal.{png,json}')